# Simulate and differentiate the same circuit on two QASP devices

Compare IQPSimulator and PauliPropagationSimulator on a commuting three-wire circuit. Check both the expectation and its Torch derivatives against default.qubit, inspect IQP auto selection, and train with both exact backends.

Use the QASP preview build in the [tutorial setup guide](../README.md), then run the cells in order.

In [ ]:
%matplotlib inline
from importlib.metadata import version
import numpy as np
import torch
import matplotlib.pyplot as plt
from IPython.display import display

print("QASP", version("qasp"))
np.random.seed(17)
torch.manual_seed(17)

## Choose a circuit that both devices can execute

Use only X-type rotations in the encoder and ansatz. The three variational words act on one, two and three wires. Z on wire zero is compatible with exact IQP tensor contraction, the Van den Nest estimator and backward Pauli propagation. A new builder for each device gives every run its own QNode.

In [ ]:
import pennylane as qml
from qasp import (AngleAnsatz, AngleEmbedding, CircuitBuilder, IQPSimulator,
                  PauliPropagationSimulator, Normal, Uniform)

n = 3
def compiled(device):
    encoder = AngleEmbedding(n, features=n, rotations="X", input_scaling=True)
    ansatz = AngleAnsatz(n, feature_map=encoder, rotations=["X 0", "XX 0 1", "XXX 0 1 2"],
                         include_imprimitive=False)
    builder = CircuitBuilder(n)
    builder.set_ansatz(ansatz)
    builder.set_measurement("expectation", observables=["Z 0"])
    builder.build()
    builder.compile(device=device, diff_method="backprop", shots=None)
    return builder

tn = compiled(IQPSimulator(n, backend="tn", mode="explicit"))
pp = compiled(PauliPropagationSimulator(n, k=None, prune=False))
reference = compiled("default.qubit")
auto = compiled(IQPSimulator(n, backend="auto", mode="explicit", seed=17))
vn = compiled(IQPSimulator(n, backend="van_den_nest", mode="explicit",
                           vn_samples=4096, seed=17))

## Initialize once and check values and gradients

Initialize both weight groups on the reference layer. Use these same numbers for all devices so differences reflect simulation. The exact PP setting retains all propagated terms. A finite weight cutoff k or coefficient pruning can reduce work and introduce approximation error.

In [ ]:
initializers = {"initialization": Normal(0, 0.4), "input_scaling_init": Uniform(0.8, 1.2)}
initial = reference.create_torchlayer(**initializers).double()
angles = initial.params.detach().clone().requires_grad_()
scales = initial.input_scaling_params.detach().clone().requires_grad_()
x = torch.tensor([0.6, -0.2, 0.3], dtype=torch.float64, requires_grad=True)

def evaluate(builder):
    value = builder.qnode(x, params=angles, input_scaling_params=scales)[0]
    grads = torch.autograd.grad(value, (angles, scales, x))
    return value.detach(), torch.cat(grads).detach()

baseline, baseline_grad = evaluate(reference)
for name, builder in [("IQP TN", tn), ("Pauli propagation", pp),
                      ("IQP auto", auto), ("IQP VN", vn)]:
    value, grad = evaluate(builder)
    print(f"{name:18s}  Z={value.item(): .6f}   max gradient error={(grad-baseline_grad).abs().max():.3g}")
    if name in {"IQP TN", "Pauli propagation"}:
        torch.testing.assert_close(value, baseline, atol=1e-8, rtol=1e-7)
        torch.testing.assert_close(grad, baseline_grad, atol=1e-8, rtol=1e-7)
print("Auto selected:", auto.device.last_report.selected)
print("VN draws / standard errors:", vn.device.last_report.samples, vn.device.last_report.standard_errors)

The sampled VN result uses classical random bitstrings. Its reported standard error describes the estimated expectation at these fixed weights. Use a larger sample count or independent seeds to study sampling variability. Auto chooses based on the supported circuit and observable; inspect last_report after execution.

## Train through the devices

Set an illustrative target for the measured moment. Each layer starts from the same angles and scales. Optimizing squared error exercises Torch backpropagation through each exact simulator.

In [ ]:
target = 0.25
trained = {}
for name, builder in [("IQP TN", tn), ("Pauli propagation", pp)]:
    layer = builder.create_torchlayer(params=angles.detach(), input_scaling_params=scales.detach()).double()
    optimizer = torch.optim.Adam(layer.parameters(), lr=0.035)
    losses = []
    for step in range(12):
        optimizer.zero_grad()
        value = layer(x.detach()).reshape(())
        loss = (value-target).square()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    trained[name] = layer
    final = layer(x.detach()).item()
    print(name, "initial loss:", losses[0], "final Z:", final)
    assert (final-target)**2 < losses[0]
    plt.plot(range(1, len(losses)+1), losses, marker=".", label=name)
plt.xlabel("Adam step")
plt.ylabel("Squared error")
plt.legend()
plt.show()

## Inspect the circuit

Use the trained weights and the same representative input for both renderers. Matplotlib is the default for the module drawing helpers.

In [ ]:
layer = trained["IQP TN"]
args = {"params": layer.params, "input_scaling_params": layer.input_scaling_params}
fig, ax = qml.draw_mpl(tn.qnode)(x.detach(), **args)
display(fig)
plt.close(fig)

### Preview the paper figure

Request standalone TikZ for a vector PDF. Symbolic labels describe the circuit architecture. Set `symbolic_parameters=False` to inspect the gate angles at this input.

`to_tikz` returns the LaTeX source. Pass it to QASP’s `show_tikz(tex)` to preview the circuit inline. The preview uses temporary files. Use `save_tikz` on the builder when you want to keep a source file.

In [ ]:
from qasp import show_tikz

tex = tn.to_tikz(x.detach(), **args, standalone=True, symbolic_parameters=True)
show_tikz(tex)

## Choose a backend for your next experiment

For a compatible IQP circuit and a few Z correlations, compare TN with VN at a stated precision and sample budget. For more general supported gates and Pauli observables, try Pauli propagation and compare an exact run against any proposed truncation. Circuit structure and readout both affect cost. For the equivalent model-dictionary options, see the [IQP device guide](../../docs/iqp_simulator.md) and [Pauli propagation guide](../../docs/pauli_propagation.md).

**Try it:** change vn_samples and repeat the fixed-point comparison. Then add a noncommuting Y rotation and compare only PP and default.qubit on that circuit.